**Simulação Computacional - Projecto Mecânico - Aula Prática 5**

<br>

# Elemento de Barra (*Truss*)

<br>

Na aula anterior aplicamos o elemento de barra elástica linear (*Truss*) a problemas no mesmo plano 2D, a generalização do elemento barra para um problema tridimencional envolve os mesmos passos que demonstramos na aula anterior, só que agora considerando mais um grau de liberdade para cada nó.

A demonstração detalhada encontra-se no capítulo 3.7 do livro *FEM First Course - Logan 5th Edition*, resultando numa matriz de rigidez, global, tridimencional, 6x6, simétrica, para o elemento de barra:

<br>

$$
k = \frac{AE}{L}
\begin{pmatrix}
C_x^{2} & C_xC_y & C_xC_z & -C_x^{2} & -C_xC_y & -C_xC_z \\
C_xC_y & C_y^{2} & C_yC_z & -C_xC_y & -C_y^{2} & -C_yC_z \\
C_xC_z & C_yC_z & C_z^{2} & -C_xC_z & -C_yC_z & -C_z^{2} \\
-C_x^{2} & -C_xC_y & -C_xC_z & C_x^{2} & C_xC_y & C_xC_z \\
-C_xC_y & -C_y^{2} & -C_yC_z & C_xC_y & C_y^{2} & C_yC_z \\
-C_xC_z & -C_yC_z & -C_z^{2} & C_xC_z & C_yC_z & C_z^{2} \\
\end{pmatrix}
$$

<br>

onde $C_x = cos\theta_x$, $C_y = cos\theta_y$ e $C_z = cos\theta_z$. Uma observação particular vem de que a estrura de $k$ tem uma simetria pelo que do ponto de vista computacional podemos simplificar a montagem da matriz:

$$
k = \frac{AE}{L}
\begin{pmatrix}
\lambda & - \lambda \\
- \lambda & \lambda \\
\end{pmatrix}
$$

onde $\lambda$ é uma matriz 3x3,

$$
\lambda =
\begin{pmatrix}
C_x^{2} & C_xC_y & C_xC_z \\
C_xC_y & C_y^{2} & C_yC_z \\
C_xC_z & C_yC_z & C_z^{2} \\
\end{pmatrix}
$$

e os cossenos directores que fazem a transformação de coordenadas dados para cada elemento,

$$
C_{x} = \frac{x_j-x_i}{L}, \quad \;
C_{y} = \frac{y_j-y_i}{L}, \quad \;
C_{z} = \frac{z_j-z_i}{L}, \quad \;
$$



---

### Exercício 3.9

Da próxima figura de elementos barra dispostos no espaço tridimencional com quatro nós, cujas coordanas são dadas em metros, pertencendo a três elementos com iguais áreas de secção ($10\times10^{-4} m^2$) e iguais módulos de elasticidade ($210GPa$). Uma carga de $20 kN$ foi aplicada no nó 1 na direção global do eixo dos $xx$. Os nós 2-4 estão fixos e impedidos de qualquer deslocamento.

![Imagem:](https://drive.google.com/uc?export=view&id=1XbORbDvQ1eSrA2ZKySmOpSlCBrjcUg9d)

<br>

Calculas os deslocamentos do nó 1. Obtém também as tensões em cada elemento.

In [ ]:
import sympy
from sympy import symbols, Matrix, sqrt, cos, acos, eye, zeros, solve

# Define as propriedades do material e variáveis simbólicas (podemos usar ou não as variáeis, escrever dá um auxílio na contagem).
A, E, AE, L1, L2, L3 = symbols('A E AE L1 L2 L3')
d1x, d1y, d1z, d2x, d2y, d2z, d3x, d3y, d3z, d4x, d4y, d4z = symbols('d1x d1y d1z d2x d2y d2z d3x d3y d3z d4x d4y d4z')
F1x, F1y, F1z, F2x, F2y, F2z, F3x, F3y, F3z, F4x, F4y, F4z = symbols('F1x F1y F1z F2x F2y F2z F3x F3y F3z F4x F4y F4z')

#-------------------------------------------------------------------------------
# 1. Definição das propriedades do material e geometria
# Define o Módulo de Elasticidade (E) e a Área da seção transversal (A).
# O produto AE é calculado, representando a rigidez axial.
E = 210 * 10**9   # Módulo de elasticidade em Pa (210 GPa)
A = 10 * 10**(-4) # Área da seção transversal em m^2 (10x10^-4 m^2)
AE = E * A        # Produto AE

# Coordenadas dos nós (x, y, z) em metros (obtidas a partir da imagem do exercício).
nodes = {
    1: (12, -3, -4),
    2: (0, 0, 0),
    3: (12, -3, -7),
    4: (14, 6, 0)
}

# Lista de elementos, definindo os nós inicial e final de cada barra.
# Os elementos são (Nó inicial, Nó final).
elements = [
    (1, 2),
    (1, 3),
    (1, 4)
]

# Inicializa a matriz de rigidez global total do problema (K_problem_global).
# Esta matriz terá dimensões 12x12, pois há 4 nós e 3 graus de liberdade (x,y,z) por nó.
K_problem_global = zeros(12, 12)


# 2. Cálculo da matriz de rigidez para cada elemento e agregação na matriz global
# Este loop itera sobre cada elemento para calcular sua matriz de rigidez local
# e agregá-la à matriz de rigidez global total.
for element_id, (node_i, node_j) in enumerate(elements):
    print(f"\n--- Elemento {element_id + 1} (do Nó {node_i} para Nó {node_j}) ---")

    # Coordenadas dos nós do elemento
    x_i, y_i, z_i = nodes[node_i]
    x_j, y_j, z_j = nodes[node_j]

    # Diferença das coordenadas entre os nós do elemento
    dx = x_j - x_i
    dy = y_j - y_i
    dz = z_j - z_i

    # Comprimento do elemento (L), calculado usando a distância euclidiana em 3D.
    L = sqrt(dx**2 + dy**2 + dz**2)
    print(f"Comprimento (L): {L.evalf(6)} m")

    # Cossenos diretores (Cx, Cy, Cz) do elemento.
    # Estes são os termos de transformação de coordenadas entre o sistema local do elemento e o sistema global.
    Cx = dx / L
    Cy = dy / L
    Cz = dz / L
    print(f"Cossenos diretores (Cx, Cy, Cz): ({Cx.evalf(6)}, {Cy.evalf(6)}, {Cz.evalf(6)})")

    # Matriz lambda (parte da matriz de rigidez local do elemento, 3x3).
    # Esta submatriz é usada para construir a matriz de rigidez global do elemento.
    k_lambda = Matrix([
        [Cx**2,   Cx*Cy,   Cx*Cz],
        [Cx*Cy,   Cy**2,   Cy*Cz],
        [Cx*Cz,   Cy*Cz,   Cz**2]
    ])

    # Matriz de rigidez global do elemento (6x6).Esta matriz representa a rigidez do elemento em coordenadas globais para os 6 graus de liberdade (3 para o nó inicial e 3 para o nó final).
    k_element_global = (AE / L) * Matrix([
        [k_lambda, -k_lambda],
        [-k_lambda, k_lambda]
    ])
    print(f"Matriz global de rigidez do elemento {element_id + 1} (6x6):")
    display(k_element_global.evalf(6))

    # Construção da matriz de rigidez global total (K_problem_global).
    # Determinamos os índices globais dos graus de liberdade para os nós (node_i e node_j) do elemento..
    global_dofs_element = []
    for node_idx in [node_i, node_j]:
        start_dof = 3 * (node_idx - 1) # Calcula o índice inicial (base 0) para o nó
        global_dofs_element.extend([start_dof, start_dof + 1, start_dof + 2])

    # Agrega a matriz de rigidez do elemento à matriz de rigidez global total.
    # Itera sobre as 6x6 entradas da matriz de rigidez do elemento e adiciona-as
    # às posições correspondentes na K_problem_global, usando os índices globais.
    for r_local in range(6):
        for c_local in range(6):
            r_global = global_dofs_element[r_local]
            c_global = global_dofs_element[c_local]
            K_problem_global[r_global, c_global] += k_element_global[r_local, c_local]

print("\n--- Matriz de Rigidez Global Total do Problema (K_problem_global, 12x12) ---")
# Exibe a matriz global total, com valores numéricos arredondados para 6 casas decimais.
display(K_problem_global.evalf(6))

# 3. Aplicação das condições de fronteira e resolução para os deslocamentos do nó 1
# De acordo com o problema, os nós 2, 3 e 4 estão fixos, o que implica que seus deslocamentos são zero.
# A carga de 20 kN é aplicada no nó 1 na direção x.
d2x = 0
d2y = 0
d2z = 0
d3x = 0
d3y = 0
d3z = 0
d4x = 0
d4y = 0
d4z = 0
F1x = 20000
F1y = 0
F1z = 0

# A matriz de rigidez é reduzida para os graus de liberdade livres do nó 1, os restantes deslocamentos são zero.
# Os graus de liberdade do nó 1 correspondem às primeiras 3 linhas e 3 colunas da K_problem_global.
K_reduced = K_problem_global[0:3, 0:3]

print("\n--- Matriz de Rigidez Reduzida (K_ff) ---")
display(K_reduced.evalf(6))

# Vetor de forças aplicadas nos graus de liberdade livres (nó 1).
F_known = Matrix([F1x, F1y, F1z])

print("\n--- Vetor de Forças Conhecidas e Aplicadas no Nó 1 (F_f) ---")
display(F_known.evalf(6))

# Vetor de deslocamentos desconhecidos (nó 1).
U_unknown = Matrix([d1x, d1y, d1z])
equations = K_reduced * U_unknown - F_known
solution_d = solve(equations, (d1x, d1y, d1z))

print("\n--- Deslocamentos Calculados para o Nó 1 ---")
# Exibe o vetor de deslocamentos calculado para o nó 1 com 6 casas decimais.
display(solution_d)

# Exibindo os resultados individualmente para maior clareza.
print(f"\nDeslocamento em x (u1x): {solution_d.get(d1x)} m")
print(f"Deslocamento em y (u1y): {solution_d.get(d1y)} m")
print(f"Deslocamento em z (u1z): {solution_d.get(d1z)} m")


--- Elemento 1 (do Nó 1 para Nó 2) ---
Comprimento (L): 13.0000 m
Cossenos diretores (Cx, Cy, Cz): (-0.923077, 0.230769, 0.307692)
Matriz global de rigidez do elemento 1 (6x6):


Matrix([
[ 1.37642e+7, -3.44106e+6, -4.58808e+6, -1.37642e+7,  3.44106e+6,  4.58808e+6],
[-3.44106e+6,    860264.0,  1.14702e+6,  3.44106e+6,   -860264.0, -1.14702e+6],
[-4.58808e+6,  1.14702e+6,  1.52936e+6,  4.58808e+6, -1.14702e+6, -1.52936e+6],
[-1.37642e+7,  3.44106e+6,  4.58808e+6,  1.37642e+7, -3.44106e+6, -4.58808e+6],
[ 3.44106e+6,   -860264.0, -1.14702e+6, -3.44106e+6,    860264.0,  1.14702e+6],
[ 4.58808e+6, -1.14702e+6, -1.52936e+6, -4.58808e+6,  1.14702e+6,  1.52936e+6]])


--- Elemento 2 (do Nó 1 para Nó 3) ---
Comprimento (L): 3.00000 m
Cossenos diretores (Cx, Cy, Cz): (0, 0, -1.00000)
Matriz global de rigidez do elemento 2 (6x6):


Matrix([
[0, 0,       0, 0, 0,       0],
[0, 0,       0, 0, 0,       0],
[0, 0,  7.0e+7, 0, 0, -7.0e+7],
[0, 0,       0, 0, 0,       0],
[0, 0,       0, 0, 0,       0],
[0, 0, -7.0e+7, 0, 0,  7.0e+7]])


--- Elemento 3 (do Nó 1 para Nó 4) ---
Comprimento (L): 10.0499 m
Cossenos diretores (Cx, Cy, Cz): (0.199007, 0.895533, 0.398015)
Matriz global de rigidez do elemento 3 (6x6):


...


--- Vetor de Forças Conhecidas e Aplicadas no Nó 1 (F_f) ---


Matrix([
[20000.0],
[      0],
[      0]])


--- Deslocamentos Calculados para o Nó 1 ---


{d1x: 0.00138372493323660, d1y: -5.15664324671658e-5, d1z: 6.01503759398499e-5}


Deslocamento em x (u1x): 0.00138372493323660 m
Deslocamento em y (u1y): -0.0000515664324671658 m
Deslocamento em z (u1z): 0.0000601503759398499 m


### 4. Cálculo das Tensões (Stress) em cada Elemento

A tensão em cada elemento de barra é calculada usando a relação constitutiva (Lei de Hooke) e os deslocamentos nodais. Para um elemento de barra, a tensão axial $\sigma$ é dada por:

$$
\sigma = \frac{F}{A}
$$

$$\sigma = \frac{E}{L} (C_x \Delta u_x + C_y \Delta u_y + C_z \Delta u_z)$$

onde $\Delta u_x = u_{j,x} - u_{i,x}$, $\Delta u_y = u_{j,y} - u_{i,y}$, $\Delta u_z = u_{j,z} - u_{i,z}$ são os deslocamentos relativos entre os nós $i$ e $j$ do elemento nas direções globais $x, y, z$. No nosso problema, o nó $i$ será sempre o nó 1, e o nó $j$ serão os nós 2, 3 ou 4. Como os nós 2, 3 e 4 estão fixos, os seus deslocamentos são zero.

In [ ]:
# Dicionário para armazenar os deslocamentos de todos os nós
# O nó 1 tem os deslocamentos calculados anteriormente, os nós 2, 3 e 4 estão fixos, então seus deslocamentos são zero.
displacements = {
    1: [solution_d.get(d1x), solution_d.get(d1y), solution_d.get(d1z)],
    2: [d2x, d2y, d2z],
    3: [d3x, d3y, d3z],
    4: [d4x, d4y, d4z]
}

print("\n--- Cálculo das Tensões nos Elementos ---")

# Itera novamente sobre os elementos para calcular a tensão em cada um
for element_id, (node_i, node_j) in enumerate(elements):
    print(f"\n--- Elemento {element_id + 1} (do Nó {node_i} para Nó {node_j}) ---")

    # Coordenadas dos nós do elemento
    x_i, y_i, z_i = nodes[node_i]
    x_j, y_j, z_j = nodes[node_j]

    # Diferença das coordenadas para calcular o comprimento e cossenos diretores
    dx = x_j - x_i
    dy = y_j - y_i
    dz = z_j - z_i

    L = sqrt(dx**2 + dy**2 + dz**2)
    Cx = dx / L
    Cy = dy / L
    Cz = dz / L

    # Deslocamentos dos nós i e j
    ui_x, ui_y, ui_z = displacements[node_i]
    uj_x, uj_y, uj_z = displacements[node_j]

    # Deslocamentos relativos
    delta_ux = uj_x - ui_x
    delta_uy = uj_y - ui_y
    delta_uz = uj_z - ui_z

    # Cálculo da tensão (sigma)
    sigma = (E / L) * (Cx * delta_ux + Cy * delta_uy + Cz * delta_uz)

    print(f"Tensão (sigma) no Elemento {element_id + 1}: {sigma.evalf(6)} Pa")


--- Cálculo das Tensões nos Elementos ---

--- Elemento 1 (do Nó 1 para Nó 2) ---
Tensão (sigma) no Elemento 1: 2.05263E+7 Pa

--- Elemento 2 (do Nó 1 para Nó 3) ---
Tensão (sigma) no Elemento 2: 4.21053E+6 Pa

--- Elemento 3 (do Nó 1 para Nó 4) ---
Tensão (sigma) no Elemento 3: -5.28941E+6 Pa


---

## Exercícios de Consolidação

Convido-vos a resolverem os problemas 3.40 da página 143 do *FEM First Course - Logan 5th Edition*.

![Imagem:](https://drive.google.com/uc?export=view&id=1HhzjQOkPINXy6l7NfSj27qE69FIs7klZ)

Para cada uma das figuras determina os deslocamentos nodais e as forças em cada elemento. Todos os elementos representados têm um E=210 GPa e um A=0.0004 $m^2$. As coordenadas na figura são dadas em metros.

---

## Abordagem da Energia Potencial Mínima para chegar à Matriz de Rigidez

O Princípio da Energia Potencial Mínima afirma que, para um corpo elástico em equilíbrio sob cargas dadas, a energia potencial total do sistema atinge um valor mínimo. A energia potencial total ($\pi_p$) de um sistema é a soma de duas componentes:

1.  **Energia de Deformação ($U$)**: A energia armazenada no material devido à sua deformação.
2.  **Energia Potencial das Cargas Externas ($\Omega$)**: O trabalho realizado pelas forças externas que atuam no sistema.

Ou seja:

$$ \pi_p = U + \Omega $$

Para o elemento de barra, vamos detalhar cada termo:

#### 1. Energia de Deformação ($U$)

Para um elemento de barra uniaxial, a energia de deformação pode ser expressa como:

$U = \int_0^L \frac{1}{2} \sigma \epsilon A \text{d}x$

Onde:
*   $\sigma$ é a tensão normal.
*   $\epsilon$ é a deformação normal.
*   $A$ é a área da seção transversal da barra.
*   $L$ é o comprimento da barra.

Usando as relações constitutivas (Lei de Hooke, $\sigma = E \epsilon$) e cinemáticas ($\epsilon = \frac{\text{d}u}{\text{d}x}$, onde $u$ é o deslocamento axial), podemos reescrever $U$ em termos do deslocamento:

$U = \int_0^L \frac{1}{2} E A (\frac{\text{d}u}{\text{d}x})^2 \text{d}x$

No contexto do MEF, o deslocamento $u(x)$ dentro do elemento é interpolado a partir dos deslocamentos nodais ($u_1$, $u_2$) usando funções de forma $N_1(x)$ e $N_2(x)$:

$u(x) = N_1(x) u_1 + N_2(x) u_2 = \mathbf{N} \mathbf{d}^e$

Onde $\mathbf{d}^e = \begin{pmatrix} u_1 \\ u_2 \end{pmatrix}$ são os deslocamentos nodais do elemento.

Então, a deformação é:

$\epsilon = \frac{\text{d}u}{\text{d}x} = \frac{\text{d}N_1}{\text{d}x} u_1 + \frac{\text{d}N_2}{\text{d}x} u_2 = \mathbf{B} \mathbf{d}^e$

Substituindo na expressão de $U$:

$U = \frac{1}{2} (\mathbf{d}^e)^T \left( \int_0^L A E \mathbf{B}^T \mathbf{B} \text{d}x \right) \mathbf{d}^e$

O termo dentro do parênteses é a matriz de rigidez do elemento, $\mathbf{k}^e$.

Para o elemento de barra com dois nós e funções de forma lineares ($N_1 = 1 - \frac{x}{L}$, $N_2 = \frac{x}{L}$), temos $\mathbf{B} = \left[ -\frac{1}{L} \quad \frac{1}{L} \right]$.

Assim, $\mathbf{k}^e = \frac{AE}{L} \begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix}$.

E a energia de deformação se torna: $U = \frac{1}{2} (\mathbf{d}^e)^T \mathbf{k}^e \mathbf{d}^e$.

#### 2. Energia Potencial das Cargas Externas ($\Omega$)

Esta energia é o negativo do trabalho realizado pelas forças externas. Para um elemento de barra, as forças externas são tipicamente as forças nodais ($f_1$, $f_2$) que agem nos nós do elemento:

$\Omega = - (f_1 u_1 + f_2 u_2) = - (\mathbf{d}^e)^T \mathbf{f}^e$

Onde $\mathbf{f}^e = \begin{pmatrix} f_1 \\ f_2 \end{pmatrix}$ são as forças nodais do elemento.

#### 3. Energia Potencial Total ($\Pi$)

Combinando os termos, a energia potencial total do elemento é:

$\Pi = \frac{1}{2} (\mathbf{d}^e)^T \mathbf{k}^e \mathbf{d}^e - (\mathbf{d}^e)^T \mathbf{f}^e$

#### 4. Princípio da Energia Potencial Mínima

Para encontrar os deslocamentos de equilíbrio que minimizam a energia potencial total, derivamos $\Pi$ em relação a cada deslocamento nodal e igualamos a zero:

$\frac{\partial \Pi}{\partial \mathbf{d}^e} = 0$

Isso nos leva às equações do elemento:

$\mathbf{k}^e \mathbf{d}^e - \mathbf{f}^e = 0$

Ou seja:

$\mathbf{k}^e \mathbf{d}^e = \mathbf{f}^e$

Esta é a equação de equilíbrio do elemento de barra, que relaciona as forças nodais com os deslocamentos nodais através da matriz de rigidez do elemento.